In [1]:
import sys, os, random
sys.path.append(os.path.join(os.getcwd(), 'CONFOLD')) #add CONFOLD to path

import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import pandas as pd

from foldrm import Classifier, get_scores, scores
from utils import split_data # Or your stratified version if you prefer
#from ModifiedClass import  extinction_birds2 # Our new function
from datasets import extinction_birds2, extinction_birds15, ecoli

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

from timeit import default_timer as timer
from datetime import timedelta

In [2]:
random.seed(42)

In [3]:
model_template, data = ecoli()

print(data.info())

data_dummies = pd.get_dummies(data)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 336 entries, 0 to 335
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   sn      336 non-null    object 
 1   mcg     336 non-null    float64
 2   gvh     336 non-null    float64
 3   lip     336 non-null    float64
 4   chg     336 non-null    float64
 5   aac     336 non-null    float64
 6   alm1    336 non-null    float64
 7   alm2    336 non-null    float64
 8   label   336 non-null    object 
dtypes: float64(7), object(2)
memory usage: 23.8+ KB
None


In [4]:
X = data[model_template.attrs].drop('label',axis=1)
X = X.convert_dtypes()
X = X.to_numpy()


y = data['label'].to_numpy()

train_data = np.concatenate((np.array(X), np.array(y).reshape(-1, 1)), axis=1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42)

In [5]:
baseline_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
train_data = np.concatenate((np.array(X_train), np.array(y_train).reshape(-1, 1)), axis=1).tolist()
test_data = np.concatenate((np.array(X_test), np.array(y_test).reshape(-1, 1)), axis=1).tolist()

In [6]:
model_template.fit(train_data, ratio=0.5)
model_template.print_asp(simple=True)
Y = [d[-1] for d in test_data]
Y_test_hat = model_template.predict(test_data)
acc = get_scores(Y_test_hat, test_data)
print('% acc', round(acc, 4), '# rules', len(model_template.crs))
acc, p, r, f1 = scores(Y_test_hat, Y, weighted=True)
print('% acc', round(acc, 4), 'macro p r f1', round(p, 4), round(r, 4), round(f1, 4), '# rules', len(model_template.crs))


label(X,'cp') :- alm1(X,N6), N6<=0.38, not ab3(X), not ab4(X), not ab5(X). [confidence: 0.95909]
label(X,'im') :- mcg(X,N1), N1<=0.61, alm1(X,N6), N6>0.59, not ab6(X), not ab7(X), not ab8(X). [confidence: 0.92241]
label(X,'pp') :- gvh(X,N2), N2>0.58, not ab9(X), not ab10(X), not ab11(X). [confidence: 0.90816]
label(X,'imu') :- mcg(X,N1), N1>0.74, alm2(X,N7), N7>0.59, not ab12(X). [confidence: 0.82692]
label(X,'cp') :- mcg(X,N1), N1<=0.52, not ab13(X), not ab15(X), not ab16(X), not ab8(X). [confidence: 0.85938]
label(X,'im') :- mcg(X,N1), N1>0.73, alm2(X,N7), N7>0.45. [confidence: 0.67857]
label(X,'im') :- alm1(X,N6), N6>0.77, alm2(X,N7), N7>0.45, not ab17(X), not ab18(X), not ab19(X). [confidence: 0.73529]
label(X,'om') :- aac(X,N5), N5>0.71, not ab20(X). [confidence: 0.78571]
label(X,'imu') :- mcg(X,N1), N1<=0.62, alm2(X,N7), N7>0.63. [confidence: 0.67857]
label(X,'imu') :- gvh(X,N2), N2<=0.53, aac(X,N5), N5<=0.7, alm2(X,N7), N7>0.63, not ab21(X), not ab23(X). [confidence: 0.76316]
la